# AraBART - three learning rates (3e-5, 5e-5, 5e-4)

Why three? The paper reports AraBART BLEU 74.79 (lr 5e-4) but the authors' stored result file says 62.49, and the repo has scripts for three learning rates. This sweep shows which learning rate gives which score.
**Settings:** GPU T4 x2, internet ON. Run cells one at a time. Total time about 20 min.

## 1. Get the code

In [ ]:
import os, glob, subprocess

# >>> EDIT THIS LINE: your GitHub username <<<
REPO_URL = "https://github.com/YOUR_USERNAME/Reproduction-of-domain-specific-Tunisian-Arabic-dialect-to-MSA-translation.git"

WORK = "/kaggle/working"
name = REPO_URL.rstrip("/").removesuffix(".git").split("/")[-1]
path = os.path.join(WORK, name)
if not os.path.exists(path):
    subprocess.run(["git", "clone", REPO_URL, path], check=True)
else:
    subprocess.run(["git", "-C", path, "checkout", "--", "."], check=True)
    subprocess.run(["git", "-C", path, "pull"], check=True)
hits = sorted(glob.glob(path + "/**/src/train_seq2seq.py", recursive=True), key=len)
assert hits, "src/train_seq2seq.py not found: check REPO_URL and that you pushed the files"
os.chdir(os.path.dirname(os.path.dirname(hits[0])))
print("working directory:", os.getcwd())
assert os.path.exists("src/summarize.py") and os.path.exists("src/train_lora.py"), "Old code on GitHub: git add . / commit / push the newest files, then re-run this cell"
print("code is up to date")

## 2. Install libraries

In [ ]:
!pip install -q "transformers>=4.46,<5" sacrebleu pyyaml sentencepiece protobuf

## 3. Data split + safety test (must say ALL CHECKS PASSED)

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

!python -m src.data
!python -m tests.test_against_stored

## 4. DRY RUN (2 min). Send me the output if anything is red.

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

!python -m src.train_seq2seq --config configs/seq2seq_arabart.yaml --dry_run --lr 5e-4

## 5. FULL SWEEP - only after the dry run worked (3 runs, about 15-20 min)

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

!python -m src.train_seq2seq --config configs/seq2seq_arabart.yaml --sweep

## 6. SUMMARY - send me this table as text

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

!python -m src.summarize results/table7_seq2seq

## 7. Package results (optional)

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

import shutil
!zip -qr results_arabart.zip results
shutil.copy("results_arabart.zip", "/kaggle/working/results_arabart.zip")